# Experimento 1. Transporte: pluma gaussiana, núcleo de la PINN y puffs

La inversión no ve las emisiones: ve una matriz que dice cuánto NOₓ pone cada fuente en cada estación y en cada píxel de TEMPO. Esa matriz es el **transporte**. Aquí se comparan cuatro, con los mismos datos de entrada (noviembre–diciembre de 2023, tardes) y las mismas constantes físicas (viento mínimo de 2 m/s; vida del NOₓ de 12 h donde el transporte la tiene):

| Transporte | Qué es | Modelo del repositorio |
|---|---|---|
| **Pluma gaussiana** | Estado estacionario: el viento de cada hora sopla para siempre y no hay memoria de las horas anteriores. Sin pérdida química. Ancho de 30° de incertidumbre en la dirección | línea base (`inversion_plume`) |
| **Núcleo de la PINN** | También estacionario, pero su forma sale de la ecuación de advección–difusión que aprendió la red (con vida del NOₓ) | `pinn_plume` |
| **Puffs gaussianos** | Cada 15 min sale un puff que viaja con el viento real de las horas siguientes, hasta 24 h, y se pierde por reacción | (investigación; migrado a `nox_ml.inversion.jacobian`) |
| **Puffs con el núcleo** | Los mismos puffs, con la forma del núcleo de la red en lugar de la gaussiana | `pinn_puff` |

Todo se calcula con `nox_ml`; el cuaderno solo llama a funciones de `nox_ml.validation.report_experiments`. Necesita los datos (`dvc pull`).

In [ ]:
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import report
from nox_ml.validation import report_experiments as rx

RUNS = report.find_runs()
ROOT = RUNS.parent
inputs = load_inputs(ROOT / "data", "2023-11-01", "2023-12-31")
print(
    f"{len(inputs.hours)} horas de tarde, {inputs.n_station_rows} observaciones de estación, "
    f"{inputs.n_pixels} píxeles de TEMPO (CDMX, noviembre–diciembre de 2023)"
)

suite = rx.TransportSuite(inputs)  # usa la red incluida en el paquete: no entrena

## 1. Mapas de una tarde típica

NOₓ que producen las emisiones a priori (fuentes puntuales y las 9 regiones viales, cada una en su valor a priori) en una hora de un día hábil a las 15 h locales, con la rapidez del viento mediana de esas horas. Arriba, el campo de cada transporte en la misma escala logarítmica; abajo, lo que cada uno añade al campo de la pluma gaussiana.

In [ ]:
maps = rx.concentration_maps(suite, rx.pick_hour(inputs))
rx.plot_maps(maps)

**Lectura.**

- La pluma gaussiana solo pone NOₓ *aguas abajo* y corta a 40 km: contra el viento no hay nada, y el campo sigue la forma de las vías porque cada hora olvida la anterior.
- El núcleo de la red hace lo mismo con otra forma: difunde algo contra el viento y a los lados, así que suaviza el campo y lo ensancha. Por eso añade NOₓ en el centro de la ciudad y alrededor de las fuentes.
- Los puffs añaden **memoria**. Con viento de 1.5 m/s, el aire de la mañana sigue sobre la ciudad por la tarde, y la pluma estacionaria no lo cuenta. Los puffs gaussianos suben el centro de forma moderada y los puffs con el núcleo, más.
- Las cuatro columnas del mapa de arriba tienen el mismo orden de magnitud: el transporte cambia cuánto y dónde, no si hay NOₓ.

## 2. ¿Conservan la masa?

Un transporte que pierde o inventa masa sin razón sesga cualquier emisión estimada. Dos pruebas:

1. **Flujo a través de una línea** perpendicular a un viento estable de 3 m/s, a distintas distancias de una fuente de 1 g/s. Una pluma sin pérdida lleva 1 g/s a cualquier distancia; con vida del NOₓ de 12 h se espera `exp(−x / uτ)`. La ecuación que aprendió la red tiene además difusión y una fuente de 1.5 km, así que su solución exacta se separa un poco de ese decaimiento: la red debe coincidir con **la solución exacta**.
2. **Masa de un puff** de cada edad, integrada en una malla alrededor de su centro, entre la que debe tener (lo emitido en 15 min por lo que sobrevive a su edad).

In [ ]:
flux = rx.crosswind_flux(suite.kernel)
flux.round(3)

In [ ]:
mass = rx.puff_mass_check(suite.kernel)
mass.round(3)

In [ ]:
rx.plot_mass(flux, mass)

**Lectura.**

- La pluma gaussiana lleva el 100 % de la masa a todas las distancias: conserva, y no tiene reacción química (por diseño).
- El núcleo pierde masa como lo pide su ecuación: a 45 km queda cerca del 68 %, frente al 69 % de la solución exacta (2–3 puntos de diferencia, el error de la red). El decaimiento `exp(−x / uτ)` solo es una referencia: la difusión y la fuente suavizada de la ecuación lo desplazan unos puntos.
- Los dos tipos de puff conservan su masa (cociente 1.00 en todas las edades): la gaussiana está normalizada, y el perfil del núcleo se normaliza a masa unitaria por construcción.

## 3. Cuánto cambia la matriz de la inversión

La inversión usa la matriz hora por hora (`J`). Si dos transportes dan la misma `J`, dan la misma emisión. Se calcula la `J` de cada transporte en una de cada 23 horas del periodo (unas 16 horas; tarda cerca de un minuto) y se compara con la de la pluma gaussiana, columna por columna: las 4 fuentes puntuales y las 9 regiones viales, en los receptores de estación (ppb por t/año) y de TEMPO (moléculas/cm²).

- `correlation`: qué tanto coinciden las entradas (1 = misma forma).
- `median_ratio`: cociente mediano entre las entradas donde ambas son positivas.
- `total_ratio`: cociente de las sumas (cuánto NOₓ en total).

In [ ]:
stacks = rx.sample_jacobians(suite)
differences = rx.jacobian_differences(stacks, rx.column_labels(inputs))
rx.difference_summary(differences).round(2)

In [ ]:
rx.plot_jacobian_differences(stacks, differences)

**Lectura.**

- **Núcleo de la PINN.** Es el más parecido a la pluma en TEMPO (correlación 0.97, total 0.93 veces) y algo menos en estaciones (0.81; total 1.2 veces, porque reparte masa en más lugares con valores más bajos). Cambia la forma más que el nivel.
- **Puffs gaussianos.** Suman en las estaciones 1.35 veces lo que la pluma (la memoria de las horas anteriores) con la misma forma general (correlación 0.90).
- **Puffs con el núcleo.** Es el que más se aleja: en estaciones suman 2.1 veces la pluma y la correlación baja a 0.71. Frente a los puffs gaussianos son 1.6 veces más, en línea con lo que dice el ADR 0005: entre 5 y 20 km los dos perfiles coinciden dentro de un factor de 1.7.
- Las regiones viales 0 y 1 son las que más cambian de un transporte a otro (el cociente mediano llega a 2.5–2.9 con los puffs con el núcleo); las fuentes puntuales cambian menos.

Consecuencia para la inversión: con el mismo dato observado, un transporte que da el doble de NOₓ por tonelada estima la mitad de emisión. Por eso el **nivel absoluto depende del transporte** (ADR 0004: +40 % de emisión vial con la PINN) y los **cocientes entre tipos de día no**: lo que cambia de forma, y no de nivel, se cancela al dividir.

## 4. Error del núcleo frente a la solución exacta

La red se entrenó solo con la ecuación, sin datos. Su error se mide contra la solución exacta de la misma ecuación (función de Bessel convolucionada con la fuente) para cuatro valores de `γ = L / (u τ)` (el cociente entre la longitud de referencia de 10 km y lo que recorre el aire en una vida del NOₓ). Aquí, `median_rel_error` y `p90_rel_error` son el error relativo mediano y del percentil 90 donde la pluma vale más del 5 % de su máximo, y `mass_error` el error de la masa total de la ventana.

In [ ]:
rx.kernel_validation(suite.kernel).round(3)

In [ ]:
rx.plot_kernel_error(suite.kernel)

**Lectura.** El error mediano es de 1.7–4.7 % según `γ` (la investigación reportó 3.5–6.6 %), el del percentil 90 llega a 12 % y la masa de la ventana se desvía 4 % o menos. El error crece hacia el borde de la ventana, aguas abajo y a los lados, donde los valores ya son pequeños; cerca de la fuente y en el eje de la pluma es de unos pocos por ciento. Es un error mucho menor que la diferencia entre transportes de la sección anterior, así que lo que separa a los modelos no es la precisión de la red sino la física que representan.